# Tutorial 3 — Scoring predictions against the gold standard

Every method in the benchmark is scored the same way:

1. its output is a binary mask at the native 2,448 x 2,048 px resolution;
2. the mask and the gold-standard mask are split into objects
   (8-connected components of at least 3 px);
3. predicted and gold-standard objects are paired one-to-one. A pair is **eligible**
   if the centroid distance is at most `d_max = 20 px` **or** the mask IoU is at least
   `IoU_min = 0.1` (the OR policy used for every published number); eligible pairs are
   assigned at minimum total cost `C = alpha (1 - IoU) + (1 - alpha) d / d_max`,
   `alpha = 0.5`;
4. matched pairs are true positives, unmatched gold-standard objects false negatives,
   and predictions with no eligible partner false positives. A prediction that had an
   eligible partner but lost the assignment is *ignored* (neither TP nor FP).
5. Object-level F1 = 2TP / (2TP + FP + FN), with TP, FP and FN **pooled over all
   images** before the ratio is taken.

This notebook applies those steps to one image with the package's own functions,
then reproduces a pooled summary from per-image rows.

In [ ]:
import os, sys, time
from pathlib import Path

def find_repo_root(start=Path.cwd()):
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists() and (p / "configs" / "default.yaml").exists():
            return p
    raise RuntimeError("Run this notebook from inside the ecdna-bench repository.")

REPO = find_repo_root()
# Where you downloaded the BioImage Archive files (the folder that contains images/).
DATA_ROOT = Path(os.environ.get("ECDNA_DATA_ROOT", Path.home() / "ecdna_data")).expanduser()
if (DATA_ROOT / "Files" / "images").is_dir():
    DATA_ROOT = DATA_ROOT / "Files"
HAVE_DATA = (DATA_ROOT / "images" / "gt_image").is_dir()
print("repository :", REPO)
print("data folder:", DATA_ROOT, "(found)" if HAVE_DATA else "(not found: the notebook runs on a small synthetic example)")

try:
    import ecdna_bench
    print("ecdna_bench:", Path(ecdna_bench.__file__).parent)
except ImportError:
    sys.path.insert(0, str(REPO / "src"))
    import ecdna_bench
    print("ecdna_bench imported from", REPO / "src")

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

NATIVE_SHAPE = (2048, 2448)
ANCHOR_UID = "ncih2170_facs_fish_0723_low_her2_52"   # the example image used in the paper

SUFFIXES = ("", "_pred_roi", "_predicted_roi", "_pred", "_roi", "_mask")

def find_file(folder, uid):
    """The file in `folder` named after the unique identifier (any image extension)."""
    folder = Path(folder) if folder else None
    if folder is None or not folder.is_dir():
        return None
    for suffix in SUFFIXES:
        hits = sorted(p for p in folder.rglob(uid + suffix + ".*")
                      if p.stem == uid + suffix
                      and p.suffix.lower() in {".tif", ".tiff", ".png", ".npy", ".npz"})
        if hits:
            return hits[0]
    return None

def read_image(path, color=False):
    flag = cv2.IMREAD_COLOR if color else cv2.IMREAD_UNCHANGED
    img = cv2.imread(str(path), flag)
    if img is None:
        import tifffile
        img = tifffile.imread(str(path))
    if color:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) if img.ndim == 3 else np.dstack([img] * 3)
    elif img.ndim == 3:
        img = img.max(axis=2)
    return img

def render_diamonds(points_rc, shape, radius=2):
    """Render (row, col) points as diamonds (|dy| + |dx| <= radius; 13 px for radius 2)."""
    mask = np.zeros(shape, np.uint8)
    offsets = [(dy, dx) for dy in range(-radius, radius + 1)
               for dx in range(-radius, radius + 1) if abs(dy) + abs(dx) <= radius]
    for r, c in np.asarray(points_rc, dtype=int):
        for dy, dx in offsets:
            y, x = r + dy, c + dx
            if 0 <= y < shape[0] and 0 <= x < shape[1]:
                mask[y, x] = 255
    return mask

def synthetic_example(seed=0, n=60):
    """A small stand-in image set, used only when the data folder is missing."""
    rng = np.random.default_rng(seed)
    pts = np.stack([rng.integers(800, 1250, n), rng.integers(1000, 1450, n)], 1)
    gs = render_diamonds(pts, NATIVE_SHAPE)
    rgb = np.full(NATIVE_SHAPE + (3,), 8, np.uint8)
    for r, c in pts:
        cv2.circle(rgb, (int(c), int(r)), 2, (40, 220, 60), -1)
    roi = np.zeros(NATIVE_SHAPE, np.uint8); roi[700:1350, 900:1550] = 255
    return {"uid": "synthetic_example", "rgb": rgb, "dapi": rgb[:, :, 2].copy(),
            "gs": gs, "roi": roi, "points": pts}

def load_image_set(uid=None):
    """Load RGB, DAPI, gold-standard mask, points and ROI for one image set."""
    if not HAVE_DATA:
        return synthetic_example()
    img_dir = DATA_ROOT / "images"
    if uid is None:
        uid = ANCHOR_UID if find_file(img_dir / "gt_image", ANCHOR_UID) else \
            sorted(p.stem for p in (img_dir / "gt_image").iterdir())[0]
    out = {"uid": uid}
    for key, sub, color in [("rgb", "rgb", True), ("dapi", "dapi", False),
                            ("gs", "gt_image", False), ("roi", "roi_mask", False)]:
        p = find_file(img_dir / sub, uid)
        out[key] = read_image(p, color=color) if p else None
    p = find_file(img_dir / "gt_coords", uid)
    out["points"] = np.load(p, allow_pickle=True) if p else None
    return out

In [ ]:
from ecdna_bench.evaluation.objects import objects_from_mask
from ecdna_bench.evaluation.matching import precompute_pairwise, resolve_matching_from_pairwise
from ecdna_bench.evaluation.metrics import object_metrics_from_counts

D_MAX, IOU_MIN, ALPHA = 20, 0.1, 0.5        # canonical operating point

def score(pred_mask, gs_mask, policy="OR", d_max=D_MAX, iou_min=IOU_MIN):
    pred = objects_from_mask(pred_mask, min_area=3, connectivity=8)
    gs = objects_from_mask(gs_mask, min_area=3, connectivity=8)
    pw = precompute_pairwise(pred, gs, max_precompute_dist=max(d_max, D_MAX))
    res = resolve_matching_from_pairwise(pw, d_max=d_max, min_iou=iou_min, alpha=ALPHA, policy=policy)
    m = object_metrics_from_counts(res.tp, res.fp, res.fn, ignored=res.ignored)
    return {"tp": res.tp, "fp": res.fp, "fn": res.fn, "ignored": res.ignored,
            "n_pred": len(pred), "n_gs": len(gs), "f1": m["f1"]}, res, pred, gs

## 1. One prediction mask

In [ ]:
sample = load_image_set()
uid = sample["uid"]
pred_path = find_file(DATA_ROOT / "predictions" / "eccount_peaks", uid) if HAVE_DATA else None
if pred_path is not None:
    pred_mask = (read_image(pred_path) > 0).astype(np.uint8) * 255
    print("prediction:", pred_path.relative_to(DATA_ROOT))
else:
    # stand-in prediction: the gold standard with some objects removed and shifted
    rng = np.random.default_rng(1)
    pts = np.asarray(sample["points"])[rng.random(len(sample["points"])) > 0.15]
    pred_mask = render_diamonds(pts + rng.integers(-2, 3, pts.shape), NATIVE_SHAPE)
    print("no deposited prediction found; using a stand-in")

row, res, pred_objs, gs_objs = score(pred_mask, sample["gs"])
print(row)

In [ ]:
view = np.zeros(NATIVE_SHAPE + (3,), np.uint8)
for i, j in res.matched:
    view[pred_objs[i]["mask"] > 0] = (255, 255, 255)      # matched prediction
for j in res.unmatched_gt:
    view[gs_objs[j]["mask"] > 0] = (0, 255, 255)          # missed gold-standard object
for i in res.unmatched_pred:
    view[pred_objs[i]["mask"] > 0] = (255, 140, 0)        # false positive
for i in res.ignored_pred:
    view[pred_objs[i]["mask"] > 0] = (128, 128, 128)      # ignored duplicate
ys, xs = np.nonzero(sample["gs"])
crop = view[max(0, ys.min() - 30): ys.max() + 30, max(0, xs.min() - 30): xs.max() + 30]
plt.figure(figsize=(8, 8)); plt.imshow(crop); plt.axis("off")
plt.title("white: matched | cyan: missed | orange: false positive | gray: ignored")

## 2. How much do the matching settings matter?

The published sensitivity analysis sweeps `d_max` and `IoU_min` under both policies.
For one image the same idea looks like this. Under OR, eligibility is set by the
distance criterion in practice; under AND, by the overlap criterion.

In [ ]:
import pandas as pd

grid = []
for policy in ("OR", "AND"):
    for d in (5, 10, 20, 40):
        for iou in (0.0, 0.1, 0.3, 0.5):
            r, *_ = score(pred_mask, sample["gs"], policy=policy, d_max=d, iou_min=iou)
            grid.append({"policy": policy, "d_max": d, "iou_min": iou, "f1": round(r["f1"], 3)})
pd.DataFrame(grid).pivot_table(index=["policy", "d_max"], columns="iou_min", values="f1")

## 3. Pooled versus per-image F1

The benchmark writes one row per image and model to
`<results>/or_matching/per_image_metrics.csv`. The published F1 pools TP, FP and FN
over images first; the mean of per-image F1 values is a different number.

In [ ]:
RESULTS = Path(os.environ.get("ECDNA_RESULTS", REPO / "release" / "frozen_results"))
per_image = RESULTS / "or_matching" / "per_image_metrics.csv"
if per_image.is_file():
    df = pd.read_csv(per_image)
    pooled = (df.groupby("model", observed=True)[["obj_tp", "obj_fp", "obj_fn"]].sum()
                .assign(pooled_f1=lambda t: 2 * t.obj_tp / (2 * t.obj_tp + t.obj_fp + t.obj_fn)))
    pooled["mean_per_image_f1"] = df.groupby("model", observed=True)["obj_f1"].mean()
    display(pooled.round(3))
else:
    print("no per-image table at", per_image,
          "- run the benchmark first (docs/TUTORIAL_EXTERNAL.md, section 3) or set ECDNA_RESULTS")

## 4. Compare a results folder with the paper

`scripts/verify_headline_numbers.py` checks F1, count MAE and mean signed bias at the
published precision. It recognizes the full benchmark (1,145 images) and the held-out
test split (175 images).

In [ ]:
import subprocess
out = subprocess.run([sys.executable, str(REPO / "scripts" / "verify_headline_numbers.py"),
                      "--results", str(RESULTS)], capture_output=True, text=True)
print(out.stdout or out.stderr)

## 5. Scoring a new method

Write one binary mask per image (PNG, 0 and 255, native resolution), named
`<unique_id>.png`, into a folder, then point the benchmark at it:

```bash
python scripts/prepare_local_run.py bia --bia-root ~/ecdna_data --out-dir runs/my_method --split test
# then, in runs/my_method/run_config.yaml, change the classical_masks: line under paths: to
#   classical_masks: /path/to/my_masks      (your method is reported as "Classic (after opt)")
python -m ecdna_bench.cli.benchmark --config runs/my_method/run_config.yaml \
    --models classical --skip-harmonize --output-dir runs/my_method/results
```

Or loop over your masks with `score()` above for a quick look.